In [ ]:
# 📦 Cell 0 - 安装依赖（首次运行建议开启）
# ------------------------------------------
#!pip install emoji pyarrow fastparquet

In [ ]:
# 📌 Cell 1 - 导入依赖库
# ------------------------------------------
import os
import re
import glob
import json
import math
import unicodedata
from datetime import datetime, timezone

import pandas as pd
from tqdm import tqdm


In [ ]:
# 📁 Cell 2 - Locate project root & locate 02 parquet inputs
# ---------------------------------------------------------
# 优先找：
#   assets/02_comment_scrape/*.parquet
# 如果没找到，就退回找：
#   Data/data_02_*/comments/*.parquet（兼容老结构）

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
ASSETS_ROOT = os.path.join(PROJECT_ROOT, "assets")
DATA_ROOT = os.path.join(PROJECT_ROOT, "Data")

# 方案 A：优先 assets/02_comment_scrape
ASSETS_02_DIR = os.path.join(ASSETS_ROOT, "02_comment_scrape")
TOP_PATH_A = os.path.join(ASSETS_02_DIR, "top_level_comments.parquet")
REP_PATH_A = os.path.join(ASSETS_02_DIR, "replies.parquet")

# 方案 B：兼容老的 Data/data_02_*
data_02_dirs = sorted(glob.glob(os.path.join(DATA_ROOT, "data_02_*")), reverse=True)
TOP_PATH_B = None
REP_PATH_B = None
if data_02_dirs:
    latest_data_02 = data_02_dirs[0]
    comment_dir = os.path.join(latest_data_02, "comments")
    # 尝试在老目录结构里找 parquet（如果你曾经这样保存过）
    TOP_PATH_B = os.path.join(comment_dir, "top_level_comments.parquet")
    REP_PATH_B = os.path.join(comment_dir, "replies.parquet")

# 选择实际存在的输入
if os.path.exists(TOP_PATH_A) and os.path.exists(REP_PATH_A):
    TOP_PATH = TOP_PATH_A
    REP_PATH = REP_PATH_A
    INPUT_MODE = "assets/02_comment_scrape"
elif TOP_PATH_B and os.path.exists(TOP_PATH_B) and REP_PATH_B and os.path.exists(REP_PATH_B):
    TOP_PATH = TOP_PATH_B
    REP_PATH = REP_PATH_B
    INPUT_MODE = "Data/data_02_*/comments"
else:
    raise FileNotFoundError(
        "❌ 找不到 02 的 parquet 输入文件。\n"
        "请确认以下路径至少有一套存在：\n"
        f"  A) {TOP_PATH_A} 和 {REP_PATH_A}\n"
        f"  B) Data/data_02_*/comments/top_level_comments.parquet 和 replies.parquet\n"
    )

print("✅ PROJECT_ROOT:", PROJECT_ROOT)
print("✅ INPUT_MODE:", INPUT_MODE)
print("✅ TOP_PATH:", TOP_PATH)
print("✅ REP_PATH:", REP_PATH)


In [ ]:
# 📦 Cell 3 - Create data_03 output folders
# ----------------------------------------
# 输出仍然放到 Data/data_03_时间戳 下面（你原来的风格）
# 同时也会额外写一份到 assets/03_cleaned（更方便后续读取）

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_03_DIR = os.path.join(DATA_ROOT, f"data_03_{timestamp}")

CLEAN_DIR = os.path.join(DATA_03_DIR, "cleaned_comments")
REPORT_DIR = os.path.join(DATA_03_DIR, "reports")

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

ASSETS_03_DIR = os.path.join(ASSETS_ROOT, "03_cleaned")
os.makedirs(ASSETS_03_DIR, exist_ok=True)

print("✅ data_03:", DATA_03_DIR)
print("✅ cleaned_comments:", CLEAN_DIR)
print("✅ reports:", REPORT_DIR)
print("✅ assets/03_cleaned:", ASSETS_03_DIR)


In [ ]:
# 📥 Cell 4 - Load parquet tables
# -------------------------------
# 你 02 输出里：
# - top_level_comments.parquet：parent 文本
# - replies.parquet：reply 文本（后续 NLI 需要）
# 清洗只需要这两张表，其余不需要清洗

top_df = pd.read_parquet(TOP_PATH)
rep_df = pd.read_parquet(REP_PATH)

print("✅ top_level_comments shape:", top_df.shape)
print("✅ replies shape:", rep_df.shape)

top_df.head(2)

In [ ]:
# ✅ Cell 5 - 基础字段检查
# ------------------------
# 如果这里报错，说明你 02 输出字段名和我们预期不一致
# 你可以把缺失字段名贴给我，我帮你对齐

TOP_REQ = ["comment_id", "thread_id", "video_id", "cell_id", "text_original"]
REP_REQ = ["comment_id", "thread_id", "parent_comment_id", "video_id", "cell_id", "text_original"]

missing_top = [c for c in TOP_REQ if c not in top_df.columns]
missing_rep = [c for c in REP_REQ if c not in rep_df.columns]

if missing_top:
    raise ValueError(f"❌ top_level_comments 缺字段: {missing_top}")
if missing_rep:
    raise ValueError(f"❌ replies 缺字段: {missing_rep}")

# 统一把 text_original 转成字符串（避免 None）
top_df["text_original"] = top_df["text_original"].astype(str).fillna("")
rep_df["text_original"] = rep_df["text_original"].astype(str).fillna("")

print("✅ Schema check passed.")


In [ ]:
# 🧼 Cell 6 - 定义清洗规则（NLI/Topic 友好）
# ------------------------------------------
# 重要原则：
# 1) 不要把 reply 清洗成空（否则 NLI 样本会塌）
# 2) URL / @mention / emoji 不直接删，而是变成占位符：
#    <URL> / <USER> / <EMOJI>
# 3) text_model 给模型：轻清洗（保留语义）
# 4) text_display 给展示：更好看（可以去掉 URL，截断）

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
USER_RE = re.compile(r"@\w[\w\.\-]{1,}", flags=re.UNICODE)

EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "]+",
    flags=re.UNICODE
)

def nfkc(text: str) -> str:
    """统一字符形态（比如全角半角）"""
    return unicodedata.normalize("NFKC", text)

def squeeze_spaces(text: str) -> str:
    """把多个空格/换行压成 1 个空格"""
    return re.sub(r"\s+", " ", text).strip()

def clean_text_model(text: str) -> str:
    """
    给模型用的轻清洗：
    - 保留标点/否定/语气
    - URL/@/emoji 变占位符
    """
    t = nfkc(text or "")
    t = URL_RE.sub(" <URL> ", t)
    t = USER_RE.sub(" <USER> ", t)
    t = EMOJI_RE.sub(" <EMOJI> ", t)
    t = t.lower()
    t = squeeze_spaces(t)
    return t

def clean_text_display(text: str, max_chars: int = 280) -> str:
    """
    给展示用的清洗：
    - URL 直接移除
    - 空格压缩
    - 截断到 max_chars
    """
    t = nfkc(text or "")
    t = URL_RE.sub(" ", t)
    t = squeeze_spaces(t)
    if max_chars and len(t) > max_chars:
        t = t[:max_chars].rstrip() + "…"
    return t

def has_url(text: str) -> bool:
    return bool(URL_RE.search(text or ""))

def has_user(text: str) -> bool:
    return bool(USER_RE.search(text or ""))

def has_emoji(text: str) -> bool:
    return bool(EMOJI_RE.search(text or ""))


In [ ]:
# 🧪 Cell 7 - 执行清洗
# --------------------
# 我们不“删除行”，而是打标：
# - is_empty_after_clean：清洗后长度太短
# 后续 04/07 如果想过滤，再基于这个字段过滤即可（更可控）

def apply_cleaning(df: pd.DataFrame, is_reply: bool) -> pd.DataFrame:
    out = df.copy()

    # 原始特征统计（用于报告）
    out["has_url_raw"] = out["text_original"].apply(has_url)
    out["has_user_raw"] = out["text_original"].apply(has_user)
    out["has_emoji_raw"] = out["text_original"].apply(has_emoji)

    # 两份文本
    out["text_model"] = out["text_original"].apply(clean_text_model)
    out["text_display"] = out["text_original"].apply(clean_text_display)

    # 长度
    out["text_len_raw"] = out["text_original"].astype(str).str.len()
    out["text_len_model"] = out["text_model"].astype(str).str.len()

    # 空文本门禁（reply 更宽松）
    min_len = 1 if is_reply else 2
    out["is_empty_after_clean"] = (out["text_len_model"] < min_len).astype(int)

    return out

top_clean = apply_cleaning(top_df, is_reply=False)
rep_clean = apply_cleaning(rep_df, is_reply=True)

print("✅ top_clean:", top_clean.shape)
print("✅ rep_clean:", rep_clean.shape)

top_clean[["text_original","text_model","text_display","has_url_raw","has_emoji_raw","has_user_raw","is_empty_after_clean"]].head(3)


In [ ]:
# 🧹 Cell 8 - 去重策略（温和）
# --------------------------
# 只做最安全的去重：
# - top：按 (video_id, comment_id)
# - replies：按 (video_id, comment_id)
# 不跨视频去重（同一句话在不同视频出现，语境不同，不能删）

top_before = len(top_clean)
rep_before = len(rep_clean)

top_clean = top_clean.drop_duplicates(subset=["video_id", "comment_id"]).copy()
rep_clean = rep_clean.drop_duplicates(subset=["video_id", "comment_id"]).copy()

print(f"✅ top dedup: {top_before} -> {len(top_clean)}")
print(f"✅ rep dedup: {rep_before} -> {len(rep_clean)}")


In [ ]:
# 📊 Cell 9 - 生成清洗报告（Quality Gate）
# ---------------------------------------
# 这里我们输出你关心的比例 + 关键新增指标：
# - empty_after_clean_rate：清洗后变空的比例（对 NLI 很关键）
# - dup_rate：去重比例

def summarize(df: pd.DataFrame, name: str) -> Dict[str, Any]:
    return {
        "name": name,
        "rows": int(len(df)),
        "url_rate": float(df["has_url_raw"].mean()) if len(df) else 0.0,
        "emoji_rate": float(df["has_emoji_raw"].mean()) if len(df) else 0.0,
        "user_rate": float(df["has_user_raw"].mean()) if len(df) else 0.0,
        "empty_after_clean_rate": float(df["is_empty_after_clean"].mean()) if len(df) else 0.0,
        "median_len_model": float(df["text_len_model"].median()) if len(df) else 0.0,
        "median_len_raw": float(df["text_len_raw"].median()) if len(df) else 0.0,
    }

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "input_mode": INPUT_MODE,
    "input_files": {"top_level_comments": TOP_PATH, "replies": REP_PATH},
    "outputs": {
        "cleaned_top_level_comments_data03": os.path.join(CLEAN_DIR, "cleaned_top_level_comments.parquet"),
        "cleaned_replies_data03": os.path.join(CLEAN_DIR, "cleaned_replies.parquet"),
        "cleaned_top_level_comments_assets": os.path.join(ASSETS_03_DIR, "cleaned_top_level_comments.parquet"),
        "cleaned_replies_assets": os.path.join(ASSETS_03_DIR, "cleaned_replies.parquet"),
        "cleaning_report": os.path.join(REPORT_DIR, "cleaning_report.json")
    },
    "summary": {
        "top_level": summarize(top_clean, "top_level"),
        "replies": summarize(rep_clean, "replies")
    }
}

# 你喜欢的“质量门禁输出风格”
print("\n" + "="*50)
print("🚦 QUALITY GATE PASSED: NLP-READY ASSETS DELIVERED")
print("="*50)

print("📊 核心指标速览：")
print(f"  - Top-level 包含 URL 比例: {report['summary']['top_level']['url_rate']*100:.1f}%")
print(f"  - Top-level 包含 Emoji 比例: {report['summary']['top_level']['emoji_rate']*100:.1f}%")
print(f"  - Replies 包含 User(@) 比例: {report['summary']['replies']['user_rate']*100:.1f}%")
print(f"  - Replies 包含 Emoji 比例: {report['summary']['replies']['emoji_rate']*100:.1f}%")
print(f"  - Replies 清洗后为空比例(重要): {report['summary']['replies']['empty_after_clean_rate']*100:.1f}%")


In [ ]:
# 🔍 Cell 11 - 快速抽样检查
# ------------------------
# 你可以随机看看几条，确认：
# - URL/@/emoji 是否变成占位符（模型输入更稳）
# - display 是否好看（dashboard 更友好）
# - 没有把否定/语气符号全删光

sample_top = top_clean.sample(n=min(5, len(top_clean)), random_state=42)
sample_rep = rep_clean.sample(n=min(5, len(rep_clean)), random_state=42)

print("\n=== Top-level sample ===")
for _, r in sample_top.iterrows():
    print("RAW:", r["text_original"][:120])
    print("MODEL:", r["text_model"][:120])
    print("DISPLAY:", r["text_display"][:120])
    print("-"*60)

print("\n=== Replies sample ===")
for _, r in sample_rep.iterrows():
    print("RAW:", r["text_original"][:120])
    print("MODEL:", r["text_model"][:120])
    print("DISPLAY:", r["text_display"][:120])
    print("-"*60)
